# Block 5 · Pattern discovery & text mining
> Data Mining · SGH Warsaw School of Economics (SMMD-ADA)
> Course anchor dataset: retail-credit / PD portfolio (+ two Block 5 add-ons)

**Business framing.** Two new datasets about the same 4,000 customers: which of ten **products** each holds (association rules — which products travel together?), and 1,200 labelled **complaints** (text mining — route each to the right team). One unsupervised craft, one supervised comeback: the multiclass task promised in Block 1.

**Learning goals for this lab**
1. Mine association rules; read support, confidence, lift — and dodge the confidence trap.
2. Run the sobering shuffled-basket check.
3. Turn text into a feature matrix: counts → TF-IDF → n-grams.
4. Train, audit, and *operate* a complaint router (confidence bar).

**How to work.** Run cells top to bottom. Before you trust any result: **Kernel → Restart & Run All.**

In [ ]:
import sys
sys.path.append("../lecture_1")   # shared course data loaders

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.max_columns', 30)

from finance_data import load_baskets, load_complaints
baskets = load_baskets()
complaints = load_complaints()
print("baskets:", baskets.shape, "| complaints:", complaints.shape)
baskets.head(3)

## 1. Mining the product baskets
Penetration first (single-item support), then frequent itemsets with Apriori, then rules.

In [ ]:
baskets.mean().sort_values(ascending=False).round(3)

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

freq = apriori(baskets, min_support=0.03, use_colnames=True)
rules = association_rules(freq, metric="confidence", min_threshold=0.3)
rules["label"] = (rules["antecedents"].apply(lambda s: " + ".join(sorted(s)))
                  + " → "
                  + rules["consequents"].apply(lambda s: " + ".join(sorted(s))))
print(f"{len(freq)} frequent itemsets, {len(rules)} rules")
rules.nlargest(8, "lift")[["label", "support", "confidence", "lift"]].round(3)

Read the top rule aloud: *17% of customers hold both mortgage and home insurance; 85% of mortgage holders have the insurance — 3.7× more often than a random customer.* The actionable list is the rule's **complement**: mortgage holders *without* insurance.

Notice the redundancy: one mortgage–insurance affinity, many decorated variants. Deduplicate before reporting.

### 1a. The confidence trap
Sort by confidence instead of lift and watch what floats up:

In [ ]:
single = rules[(rules["antecedents"].apply(len) == 1)
               & (rules["consequents"].apply(len) == 1)]
single.nlargest(5, "confidence")[["label", "support", "confidence", "lift"]].round(3)

`savings → checking` at 92.7% confidence looks spectacular — until you check that 92.4% of *everyone* holds a checking account. Lift ≈ 1.00: pure base-rate flattery. **Filter by support, rank by lift, read confidence for the campaign promise.**

### 1b. The sobering check: rules on shuffled baskets
Shuffle every column independently — all penetrations survive, all co-occurrence dies. Then mine with identical settings:

In [ ]:
rng = np.random.default_rng(0)
shuffled = pd.DataFrame({c: rng.permutation(baskets[c].values)
                         for c in baskets.columns}, index=baskets.index)

freq_s = apriori(shuffled, min_support=0.03, use_colnames=True)
rules_s = association_rules(freq_s, metric="confidence", min_threshold=0.3)
print(f"real baskets:     {len(rules)} rules, best lift {rules['lift'].max():.2f}")
print(f"shuffled baskets: {len(rules_s)} rules, best lift {rules_s['lift'].max():.2f}")

Hundreds of "rules" pass the same filters on pure noise — the rule *count* is meaningless. What noise cannot fake is **lift**: ~1.1 versus 4.28. Demand lift that clears noise levels before calling anything an insight.

## 2. Text becomes numbers

In [ ]:
complaints.sample(3, random_state=1)[["category", "text"]]

In [ ]:
# Bag of words + Zipf's law
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
counts = cv.fit_transform(complaints["text"])
print(f"matrix: {counts.shape[0]} docs × {counts.shape[1]} words, "
      f"{counts.nnz / np.prod(counts.shape):.1%} non-zero (sparse!)")

freqs = np.sort(np.asarray(counts.sum(0)).ravel())[::-1]
plt.figure(figsize=(6, 3))
plt.loglog(range(1, len(freqs) + 1), freqs)
plt.xlabel("word rank (log)"); plt.ylabel("frequency (log)"); plt.title("Zipf")
plt.tight_layout(); plt.show()

A few words are everywhere, most are rare — every corpus looks like this. `min_df` / `max_df` prune both useless ends. Now weigh the survivors by distinctiveness (TF-IDF) and check what idf thinks of a few words:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer()
tv.fit(complaints["text"])
vocab = {w: i for i, w in enumerate(tv.get_feature_names_out())}
for w in ["account", "bank", "mortgage", "skimmed", "casino"]:
    print(f"idf({w:10s}) = {tv.idf_[vocab[w]]:.2f}")

## 3. The complaint router
TF-IDF + logistic regression, in a pipeline (the vectoriser's vocabulary and idf are **fitted parameters** — train data only, as always).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_tr, X_te, y_tr, y_te = train_test_split(
    complaints["text"], complaints["category"], test_size=0.3,
    random_state=42, stratify=complaints["category"])

router = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))
router.fit(X_tr, y_tr)
pred = router.predict(X_te)
print(f"test accuracy: {accuracy_score(y_te, pred):.3f}  (chance: 0.20)")
print(classification_report(y_te, pred, digits=2))

In [ ]:
cats = sorted(y_te.unique())
cm = confusion_matrix(y_te, pred, labels=cats)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(5)); ax.set_xticklabels(cats, rotation=30, ha="right")
ax.set_yticks(range(5)); ax.set_yticklabels(cats)
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()*0.6 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
plt.tight_layout(); plt.show()

Where do the errors live? Print a few misrouted complaints — most of them **ramble across topics** (a fees complaint that also mentions the app), exactly the cases a human router would also hesitate on:

In [ ]:
wrong = X_te[pred != y_te]
for text, true, p in list(zip(wrong, y_te[pred != y_te], pred[pred != y_te]))[:3]:
    print(f"[true: {true} | predicted: {p}]\n{text[:220]}...\n")

### 3a. Operating the router: the confidence bar
Route automatically only above a confidence bar; queue the rest for humans — Block 3's threshold logic, multiclass edition.

In [ ]:
proba = router.predict_proba(X_te)
conf = proba.max(axis=1)
for bar in [0.0, 0.5, 0.6, 0.7]:
    m = conf >= bar
    acc = (pred[m] == y_te[m]).mean()
    print(f"bar {bar:.1f}: auto-routed {m.mean():5.0%}, "
          f"accuracy among them {acc:6.1%}, human queue {1-m.mean():5.0%}")

At a 0.6 bar, four in five complaints route themselves at ~98% accuracy — and humans see only the genuinely hard fifth. The bar is a **business decision**, chosen with the operations team.

### 3b. What the model reads
Top coefficients per class (stopwords removed for display) — a sanity check you can literally read. An off-topic top word would be the leakage smell, text edition.

In [ ]:
display_model = make_pipeline(TfidfVectorizer(stop_words="english"),
                              LogisticRegression(max_iter=2000))
display_model.fit(X_tr, y_tr)
words = np.array(display_model.named_steps["tfidfvectorizer"].get_feature_names_out())
lr = display_model.named_steps["logisticregression"]
for cat, coefs in zip(lr.classes_, lr.coef_):
    top = words[np.argsort(coefs)[-5:]][::-1]
    print(f"{cat:12s}: {', '.join(top)}")

## 4. Exercises
1. **N-grams:** refit the router with `TfidfVectorizer(ngram_range=(1, 2))`. How much accuracy do bigrams buy, and how many features do they cost? Then add `min_df=3`.
2. **Naive Bayes bake-off:** swap in `MultinomialNB()` (with `CountVectorizer`). Faster? Better? Worse? The baseline discipline applies to text too.
3. **min_support sweep:** re-mine the baskets at `min_support` ∈ {0.01, 0.05, 0.15}. How does the rule count change — and at which setting would you *report* rules to a business owner?
4. **Cluster the complaints** (Block 4 meets Block 5): k-means with k=5 on the TF-IDF matrix, then cross-tabulate clusters against true categories. How well does unsupervised structure recover the labels?
5. **Char n-grams for robustness:** `TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))`. Compare accuracy — and argue why this variant survives typos (and Polish inflection) better.
6. **Break it honestly:** append the sentence "and the app crashed" to every test document. Which categories suffer, and what does that say about deploying routers on drifting text?

*Hand-in for the project team: three basket rules you would act on (all three numbers + one honest sentence each) and your router's confusion matrix with a two-sentence reading.*

**"Done" for today** = your rules survive the lift-vs-confidence interrogation, and you can explain every off-diagonal cell of your confusion matrix.

In [ ]:
# scratch cell for the exercises